# Parsing Property Price Data

This notebook parses the dataset that combines prices from June 2021 to September 2025, using originally provided property price datasets (`PriceMerge_Jun21_Sept25.csv`). The CSV was created using basic excel features to combine data across the different xlsx price files. The notebook extracts structured information such as **Town**, **Address**, **Unit Number**, and **Property Features** from the concatenated `Application Property` column.

The parsed data is later used to merge with resale data for further analysis.


### Reference
This notebook refers to the resale dataset generated in  
**`Ria_ParseResaleData.ipynb`**,  
which produced the file `ParsedResale_updatedSept2025_New.csv`.


### Pipeline Overview
1. Load datasets (price data + reference resale data)  
2. Identify the resale price column dynamically  
3. Extract property features (text in parentheses)  
4. Extract unit numbers and town names  
5. Clean address text  
6. Build and export the parsed dataset


In [ ]:
import pandas as pd
import re
from google.colab import files


# --- Function 1: Load datasets ---
def load_datasets(price_path, resale_path):
    """Load and clean input datasets."""
    df1 = pd.read_csv(price_path, dtype=str).fillna('')
    df2 = pd.read_csv(resale_path, dtype=str).fillna('')
    return df1, df2


# --- Function 2: Identify the resale price column ---
def identify_resale_column(df):
    """Identify the column containing both 'resale' and 'price' (case-insensitive)."""
    for col in df.columns:
        if "resale" in col.lower() and "price" in col.lower():
            print(f"✅ Using column '{col}' as Maximum Resale Price")
            return col
    raise KeyError("❌ Could not find a column containing both 'resale' and 'price' in Dataset 1.")


# --- Function 3: Extract property features and clean property names ---
def extract_property_features(df):
    """Extract property features (inside parentheses) and clean property names."""
    df['Property Feature'] = (
        df['Application Property']
        .str.extract(r'\(([^)]+)\)', expand=False)
        .str.strip()
    )
    df['Clean_Property'] = (
        df['Application Property']
        .str.replace(r'\([^)]*\)', '', regex=True)
        .str.strip()
    )
    return df


# --- Function 4: Extract unit numbers ---
def extract_unit_numbers(df):
    """Extract unit numbers from cleaned property strings."""
    df['Unit Number'] = df['Clean_Property'].str.extract(
        r'(?:Unit[-\s]*|#)([A-Za-z0-9\-]+)', expand=False
    )
    return df


# --- Function 5: Find towns using a reference list ---
def detect_towns(df, town_list):
    """Detect the town for each property by matching against a reference list."""
    def find_town(text):
        for town in town_list:
            pattern = r'\b' + re.escape(town) + r'\b'
            if re.search(pattern, text, flags=re.IGNORECASE):
                return town
        return ''
    df['Town'] = df['Clean_Property'].apply(find_town)
    return df


# --- Function 6: Clean address field ---
def clean_address_field(df):
    """Remove town names and unit references from the property string."""
    def clean_address(text, town, unit):
        if town:
            text = re.sub(r'\b' + re.escape(town) + r'\b', '', text, flags=re.IGNORECASE)
        text = re.sub(
            r'(?:,?\s*Unit[-\s]*[A-Za-z0-9\-]+|#\s*[A-Za-z0-9\-]+)',
            '',
            text,
            flags=re.IGNORECASE
        )
        return text.strip(' ,~-')

    df['Address'] = df.apply(
        lambda x: clean_address(x['Clean_Property'], x['Town'], x['Unit Number']),
        axis=1
    )
    return df


# --- Function 7: Build final parsed dataset ---
def build_final_dataset(df, resale_col):
    """Select relevant columns and rename for clarity."""
    final_df = df[['Town', 'Address', 'Unit Number', resale_col, 'Property Feature']].copy()
    final_df.rename(columns={resale_col: 'Maximum Resale Price'}, inplace=True)
    final_df['Notes_Dataset_Src'] = 'Dataset1'
    return final_df


# --- Function 8: Export parsed dataset ---
def export_dataset(df, output_path):
    """Save and download the parsed dataset as CSV."""
    df.to_csv(output_path, index=False)
    files.download(output_path)
    print(f"✅ File successfully saved and downloaded as: {output_path}")


# --- Main Execution Pipeline ---
def main():
    # Step 1: Load datasets
    df1, df2 = load_datasets(
        "/content/PriceMerge_Jun21_Sept25.csv",
        "/content/ParsedResale_updatedSept2025_New.csv"
    )

    # Step 2: Identify resale price column
    resale_col = identify_resale_column(df1)

    # Step 3–6: Parse and clean property information
    df1 = extract_property_features(df1)
    df1 = extract_unit_numbers(df1)
    df1 = detect_towns(df1, df2['Town'].dropna().unique().tolist())
    df1 = clean_address_field(df1)

    # Step 7: Build final structured dataset
    final_df = build_final_dataset(df1, resale_col)

    # Step 8: Export parsed CSV
    export_dataset(final_df, "/content/Jun21_Sept25_Parsed.csv")

    # Preview results
    display(final_df.head())


# --- Run the pipeline ---
if __name__ == "__main__":
    main()